### TO DO

- Remove punctuation

- Remove (), [], ;, : and etc. because they are used as emojis

- Deal with semi-space before and after words
    - مسئله‌ای, سخت‌‌تر, میمیره, می‌زنم, می‌کنند

- Remove stopwords, keep negative stopwords for content

- Tokenize 

- Check for non-Persian words after tokenization and drop them 

- Stemming



In [1]:
# Import the libraries
import pandas as pd
import os
import re
from collections import Counter

# Shekar Imports
from shekar import Normalizer
from shekar.preprocessing import (
    EmojiRemover,
    PunctuationRemover,
    StopWordRemover,
    NonPersianRemover
)
from shekar import WordTokenizer
from shekar import Stemmer
from shekar import POSTagger
from shekar import NER

In [2]:
# Load the dataset
df = pd.read_csv('../data/cleaned/all_cleaned.csv', encoding = 'utf-8-sig')
TEXT_COL = 'cleaned_text'

print(f'Loaded {len(df)} messages')
print(f'Timelines: {sorted(df['timeline'].unique())}')

df.head()

Loaded 11174 messages
Timelines: ['timeline_1', 'timeline_2', 'timeline_3']


,message_id,cleaned_text,timeline
0,129507,تنها کسی که تو مملکت داره کارش رو خوب انجام می...,timeline_1
1,129512,اولین سالی که بابانوئل برای بچه‌های ما چیزی آو...,timeline_1
2,129513,داشتم گالریمو مرتب می‌کردم عکس بچگی این تخم سگ...,timeline_1
3,129514,دم رفتن بهم گفت «اصن حواست بود کجاها امن بودم؟...,timeline_1
4,129516,تو خیابون ما ۲۰ ساله یه ساندویچی هست، صاحبش خا...,timeline_1


In [3]:
# Init all tools
normalizer = Normalizer()
emoji_remover = EmojiRemover()
punct_remover = PunctuationRemover()
stopword_remover = StopWordRemover()
non_presian_remover = NonPersianRemover()
tokenizer = WordTokenizer()
stemmer = Stemmer()

# POS and NER 
pos_tagger = POSTagger()
ner = NER()

print('Done')

Done


In [4]:
# Define negative stopwords so we wont remove them 
NEGATIVE_STOPWORDS = {
    # negation
    'نه', 'نمی', 'نیست', 'نبود', 'نشده', 'نکرد',
    # negative adjectives
    'بد', 'خراب', 'زشت', 'ضعیف', 'سخت', 'تلخ', 'وحشتناک', 'افتضاح',
    'بیچاره', 'ناراحت', 'غمگین', 'عصبانی', 'خشمگین', 'متاسف',
    # negative verbs
    'نمی‌شود', 'نمی‌کنم', 'نمی‌کنی', 'نمی‌کند', 'نمی‌کنیم', 'نمی‌کنید', 'نمی‌کنند',
    # negative nouns
    'مشکل', 'درد', 'غم', 'استرس', 'نگرانی', 'ترس', 'بحران', 'فاجعه',
    # negative adverbs
    'هرگز', 'اصلا', 'اصلاً', 'هیچ',
}

In [5]:
def remove_stopwords_keep_negatives(text):
    '''
    Remove Persian stopwords using Shekar builtin stopwods.
    Also exclude negative words that may be removed by the source library.
    '''
    if not isinstance(text, str) or text.strip() == '':
        return ''
    
    # First tokenize 
    tokens_before = list(tokenizer(text))

    # Shekar stopword remover
    text_no_stop = stopword_remover(text)
    tokens_after = list(tokenizer(text_no_stop)) if text_no_stop.strip() else []

    # Put back negative words if removed
    final_tokens = []
    for tok in tokens_before:
        if tok in NEGATIVE_STOPWORDS:
            final_tokens.append(tok)
        elif tok in tokens_after:
            final_tokens.append(tok)

    return ' '.join(final_tokens)

# Test
test_text = 'من این فیلم رو دوست ندارم و خیلی بد بود به نظرم'
print(test_text)
print(remove_stopwords_keep_negatives(test_text))

من این فیلم رو دوست ندارم و خیلی بد بود به نظرم
فیلم دوست ندارم بد بود نظرم


In [6]:
# Normalizer 
def normalize_text(text):
    '''
    Arabic -> Persian characters
    semi-space
    remove emojis if there is any left
    remove punctuation and symbols
    '''
    if not isinstance(text, str) or text.split() == '':
        return ''
    
    # Shekar normalization 
    text = normalizer(text)

    # Remove emojis
    text = emoji_remover(text)

    # Remove punct
    text = punct_remover(text)

    return text

In [7]:
# Normalize all rows
df['normalized'] = df[TEXT_COL].apply(normalize_text)

# Sample to check
for i in range(min(3, len(df))):
    print(f'--- Original ---\n{df.loc[i, TEXT_COL][:150]}')
    print(f'--- Normalized ---\n{df.loc[i, 'normalized'][:150]}')
    print()

--- Original ---
تنها کسی که تو مملکت داره کارش رو خوب انجام میده چسب زن الویه نامینو!
داداش یکم شل بگیر باز نمیشه این
--- Normalized ---
تنها کسی که تو مملکت داره کارش رو خوب انجام می‌ده چسب زن الویه نامینو
داداش یکم شل بگیر باز نمی‌شه این

--- Original ---
اولین سالی که بابانوئل برای بچه‌های ما چیزی آورد، فکر می‌کنید واکنش دخترم چی بود؟
صبح بیدار شد و هدیه‌ش رو پای درخت دید، با نگرانی برگشت گفت وای مامان
--- Normalized ---
اولین سالی که بابانوئل برای بچه‌های ما چیزی آورد فکر می‌کنید واکنش دخترم چی بود
صبح بیدار شد و هدیه‌ش رو پای درخت دید با نگرانی برگشت گفت وای مامان سن

--- Original ---
داشتم گالریمو مرتب می‌کردم عکس بچگی این تخم سگو دیدم که یه روز رندوم تصمیم گرفت دیگه نیاد پیشم
خامه اگه این توییتو می‌بینی برگرد میوزاده
--- Normalized ---
داشتم گالریمو مرتب می‌کردم عکس بچگی این تخم سگو دیدم که یه روز رندوم تصمیم گرفت دیگه نیاد پیشم
خامه اگه این توییتو می‌بینی برگرد میوزاده



In [8]:
# Remove stop words
df['text_nostop'] = df['normalized'].apply(remove_stopwords_keep_negatives)

# Sample to check
for i in range(min(3, len(df))):
    print(f'--- Normalized ---\n{df.loc[i, 'normalized'][:150]}')
    print(f'--- NoStopWords ---\n{df.loc[i, 'text_nostop'][:150]}')
    print()

--- Normalized ---
تنها کسی که تو مملکت داره کارش رو خوب انجام می‌ده چسب زن الویه نامینو
داداش یکم شل بگیر باز نمی‌شه این
--- NoStopWords ---
مملکت داره کارش انجام می‌ده چسب زن الویه نامینو داداش یکم شل بگیر نمی‌شه

--- Normalized ---
اولین سالی که بابانوئل برای بچه‌های ما چیزی آورد فکر می‌کنید واکنش دخترم چی بود
صبح بیدار شد و هدیه‌ش رو پای درخت دید با نگرانی برگشت گفت وای مامان سن
--- NoStopWords ---
سالی بابانوئل بچه‌های آورد فکر می‌کنید واکنش دخترم چی بود صبح بیدار شد هدیه‌ش پای درخت دید نگرانی برگشت گفت وای مامان سنتا اومده نکنه دزده همونجا بتون

--- Normalized ---
داشتم گالریمو مرتب می‌کردم عکس بچگی این تخم سگو دیدم که یه روز رندوم تصمیم گرفت دیگه نیاد پیشم
خامه اگه این توییتو می‌بینی برگرد میوزاده
--- NoStopWords ---
داشتم گالریمو مرتب می‌کردم عکس بچگی تخم سگو دیدم یه روز رندوم تصمیم گرفت دیگه نیاد پیشم خامه توییتو می‌بینی برگرد میوزاده



In [9]:
# Tokenize 
df['tokens'] = df['text_nostop'].apply(lambda x: list(tokenizer(x)) if x else [])

# Check the samples
for i in range(min(3, len(df))):
    tokens = df.loc[i, 'tokens']
    print(tokens[:20])

['مملکت', 'داره', 'کارش', 'انجام', 'می\u200cده', 'چسب', 'زن', 'الویه', 'نامینو', 'داداش', 'یکم', 'شل', 'بگیر', 'نمی\u200cشه']
['سالی', 'بابانوئل', 'بچه\u200cهای', 'آورد', 'فکر', 'می\u200cکنید', 'واکنش', 'دخترم', 'چی', 'بود', 'صبح', 'بیدار', 'شد', 'هدیه\u200cش', 'پای', 'درخت', 'دید', 'نگرانی', 'برگشت', 'گفت']
['داشتم', 'گالریمو', 'مرتب', 'می\u200cکردم', 'عکس', 'بچگی', 'تخم', 'سگو', 'دیدم', 'یه', 'روز', 'رندوم', 'تصمیم', 'گرفت', 'دیگه', 'نیاد', 'پیشم', 'خامه', 'توییتو', 'می\u200cبینی']


In [10]:
# Remove any non-Persian words
def is_persian_word(word):
    '''
    Return True if the word contains Persian character
    '''
    # Allow semi-space, Persian digits, and Persian letters
    persian_range = re.compile(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\u200c]')

    return bool(persian_range.search(word))

def filter_persian_tokens(tokens):
    '''Keep tokens that contain Persian characters'''
    return [w for w in tokens if is_persian_word(w)]

In [11]:
# Filter non-Persian words
before_counts = df['tokens'].apply(len).sum()
df['tokens_persian'] = df['tokens'].apply(filter_persian_tokens)
after_counts = df['tokens_persian'].apply(len).sum()

print(f'Before: {before_counts}')
print(f'After: {after_counts}')
print(f'Removed {before_counts - after_counts} non-Persian words')

Before: 202536
After: 199884
Removed 2652 non-Persian words


In [12]:
# Stemming
df['tokens_stemmed'] = df['tokens_persian'].apply(
    lambda toks: [stemmer(w) for w in toks] if toks else []
)

# Sample
for i in range(min(3, len(df))):
    persian = df.loc[i, 'tokens_persian'][:10]
    stemmed = df.loc[i, 'tokens_stemmed'][:10]
    pairs = list(zip(persian, stemmed))
    
    for orig, stem in pairs:
        print(f'    {orig} -> {stem}')
    print()

    مملکت -> مملک
    داره -> داره
    کارش -> کار
    انجام -> انجا
    می‌ده -> می‌ده
    چسب -> چسب
    زن -> زن
    الویه -> الویه
    نامینو -> نامینو
    داداش -> دادا

    سالی -> سال
    بابانوئل -> بابانوئل
    بچه‌های -> بچه
    آورد -> آورد
    فکر -> فکر
    می‌کنید -> می‌کنید
    واکنش -> واکن
    دخترم -> دختر
    چی -> چی
    بود -> بود

    داشتم -> داشت
    گالریمو -> گالریمو
    مرتب -> مرتب
    می‌کردم -> می‌کردم
    عکس -> عکس
    بچگی -> بچگی
    تخم -> تخم
    سگو -> سگو
    دیدم -> دید
    یه -> یک



In [ ]:
#aaaa